# 🚀 ACE-Net Baseline: End-to-End FakeAVCeleb (N=700) Evaluation
### Model: **Adapted ACE-Net** (`acenet_adapted.pth`)
### Benchmark: Official 700 FakeAVCeleb clips (350 Real / 350 Fake, Aligned with DeepSentinel)
### Outputs: Saved directly to Google Drive `Baseline_training/checkpoints/`

---

## Step 1: Connect to GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import torch
from pathlib import Path

drive.mount('/content/drive')

print('GPU Available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name   :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not active! Go to Runtime -> Change runtime type -> T4 GPU.')


## Step 2: Clone or Pull Baseline Repository & Install Dependencies

In [ ]:
import os
%cd /content
if not os.path.exists('/content/Baseline_Training'):
    !git clone https://github.com/gjvlio/Baseline_Training.git
%cd /content/Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!pip install -q scikit-learn transformers librosa
print('✅ Environment ready!')


## Step 3: Configure Drive Paths & Verify Aligned 700 Manifest
Points directly to `My Drive/Baseline_training` checkpoints and manifests.

In [ ]:
import os
import shutil
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/Baseline_training')
CKPT_PATH = DRIVE_BASE / 'checkpoints' / 'acenet_adapted.pth'
OUTPUT_CSV = DRIVE_BASE / 'checkpoints' / 'preds_acenet_adapted_700.csv'
LOCAL_MANIFEST = Path('Manifests/fakeavceleb_eval_700.csv')
DRIVE_MANIFEST = DRIVE_BASE / 'manifests' / 'fakeavceleb_eval_700.csv'

# Ensure aligned manifest is available
if DRIVE_MANIFEST.exists() and not LOCAL_MANIFEST.exists():
    shutil.copy(DRIVE_MANIFEST, LOCAL_MANIFEST)
    print(f'✅ Loaded aligned manifest from Drive: {DRIVE_MANIFEST}')
elif LOCAL_MANIFEST.exists() and DRIVE_BASE.exists():
    (DRIVE_BASE / 'manifests').mkdir(parents=True, exist_ok=True)
    shutil.copy(LOCAL_MANIFEST, DRIVE_MANIFEST)
    print(f'✅ Synced aligned manifest to Drive: {DRIVE_MANIFEST}')

print('=' * 75)
print('📁 CONFIGURATION OVERVIEW:')
print(f'  Checkpoint Path : {CKPT_PATH} (Exists: {CKPT_PATH.exists()})')
print(f'  Target Manifest : {LOCAL_MANIFEST} (Exists: {LOCAL_MANIFEST.exists()})')
print(f'  Output CSV Path : {OUTPUT_CSV}')
print('=' * 75)
assert CKPT_PATH.exists(), f'❌ Checkpoint hindi mahanap sa: {CKPT_PATH}'


## Step 4: Fast Selective Unzip of ONLY the 700 MP4 Videos (~15-20 Seconds)
Automatically locates the video zip in `My Drive/Baseline_training` and extracts ONLY the 700 target clips.

In [ ]:
%cd /content/Baseline_Training

# Awtomatikong hahanapin ang zip archive sa Drive directory
!python -m scripts.extract_700_videos \
    --zip-path '/content/drive/MyDrive/Baseline_training' \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --output-dir '/content/fakeav_raw'


## Step 5: Run End-to-End Evaluation & Statistical Significance Testing
Runs inference on the exact 700 clips, computes AUC/Accuracy/F1, and performs paired DeLong test vs DeepSentinel.

In [ ]:
%cd /content/Baseline_Training

# Patakbuhin ang evaluator gamit ang adapted checkpoint at aligned manifest
!python -u -m scripts.evaluate_fakeavceleb_700 \
    --manifest 'Manifests/fakeavceleb_eval_700.csv' \
    --raw-dir '/content/fakeav_raw' \
    --ckpt '/content/drive/MyDrive/Baseline_training/checkpoints/acenet_adapted.pth' \
    --output-csv '/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_adapted_700.csv' \
    --device cuda


## Step 6: Verify Generated Output Files in Google Drive

In [ ]:
import pandas as pd
from pathlib import Path

preds_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/preds_acenet_adapted_700.csv')
metrics_file = Path('/content/drive/MyDrive/Baseline_training/checkpoints/metrics_preds_acenet_adapted_700.csv')

if preds_file.exists():
    df_preds = pd.read_csv(preds_file)
    print(f'✅ Predictions saved successfully! Total rows: {len(df_preds):,}')
    display(df_preds.head(5))

if metrics_file.exists():
    df_metrics = pd.read_csv(metrics_file)
    print('\n📊 OFFICIAL COMPARATIVE METRICS:')
    display(df_metrics)
